# Construct final data

This notebook constructs the parquet files pegel_weather_6_hourly_lags12_days_weather_lags_forecast_48h.parquet and pegel_weather_6_hourly_lags12_days_weather_lags_forecast_72h.parquet for model training.

The timestamp column contains the date and hour when the Kirchentellinsfurt water level was reported.
Features include lagged weather variables, lagged Kirchentellinsfurt water levels, and lagged Horb water levels.
Horb water levels are predictors only and are not included in the forward `water_levels` targets.


In [18]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../../data/intermediate")
OUT_DIR = Path("../out")
PEGEL_PATH = DATA_DIR / "pegel_kirchentellinsfurt.csv"
HORB_PEGEL_PATH = DATA_DIR / "pegel_horb.csv"
WEATHER_HOURLY_PATH = DATA_DIR / "weather_data_hourly_1968_2026.csv"

def load_pegel(pegel_path=PEGEL_PATH, cutoff_year=1968):
    pegel = pd.read_csv(pegel_path, sep=";", usecols=["Messzeitpunkt", "Messwert"], dtype="string")
    pegel["timestamp"] = pd.to_datetime(pegel.pop("Messzeitpunkt"), format="%d.%m.%Y %H:%M", errors="coerce")
    pegel["water_level"] = pd.to_numeric(pegel.pop("Messwert").str.replace(",", ".", regex=False), errors="coerce")
    pegel = pegel.dropna(subset=["timestamp", "water_level"]).sort_values("timestamp")
    pegel["year"] = pegel["timestamp"].dt.year
    pegel["month"] = pegel["timestamp"].dt.month
    pegel = pegel[pegel["year"] >= cutoff_year]
    return pegel


def load_horb_pegel(pegel_path=HORB_PEGEL_PATH, cutoff_year=1968):
    pegel = pd.read_csv(pegel_path, sep=";", usecols=["Messzeitpunkt", "Messwert"], dtype="string")
    pegel["timestamp"] = pd.to_datetime(pegel.pop("Messzeitpunkt"), format="%d.%m.%Y %H:%M", errors="coerce")
    pegel["horb_water_level"] = pd.to_numeric(pegel.pop("Messwert").str.replace(",", ".", regex=False), errors="coerce")
    pegel = pegel.dropna(subset=["timestamp"]).sort_values("timestamp")
    pegel = pegel.sort_values("timestamp")
    pegel = pegel[pegel["timestamp"].dt.year >= cutoff_year]
    return pegel


def load_hourly_weather(weather_path=WEATHER_HOURLY_PATH):
    weather = pd.read_csv(weather_path, parse_dates=["timestamp"])
    weather["timestamp"] = weather["timestamp"].dt.floor("h")
    weather["station_id"] = weather["station_id"].astype(str).str.zfill(5)
    return weather

weather_data_hourly = load_hourly_weather()
pegel_data = load_pegel()
horb_pegel_data = load_horb_pegel()


In [19]:
# Pivot hourly weather data wide so each station-variable is a feature column.
weather_data_hourly_pivoted = weather_data_hourly.pivot(index='timestamp', columns='station_id', values=['temperature_air_2m', 'precipitation_amount', 'humidity_relative', 'pressure_vapor']).reset_index()
weather_data_hourly_pivoted.columns = ['_'.join(map(str, col)) for col in weather_data_hourly_pivoted.columns]
weather_data_hourly_pivoted.rename(columns={'timestamp_': 'timestamp'}, inplace=True)

/var/folders/bl/5wzgflnj66j7k0xyryxc3pn40000gn/T/ipykernel_10707/69002907.py:4: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  weather_data_hourly_pivoted.rename(columns={'timestamp_': 'timestamp'}, inplace=True)


In [20]:
# See if hourly weather data is always reported exactly on the hour
repoting_times = weather_data_hourly['timestamp'].dt.minute.value_counts()
print(repoting_times)

timestamp
0    1226336
Name: count, dtype: int64


In [21]:
# Add hourly lags of weather and water-level as well as daily rolling averages of weather variables to the pegel data.

LAGS = 6
DAILY_OFFSETS = 11

weather_data_hourly_pivoted = (
    weather_data_hourly_pivoted
    .sort_values("timestamp")
    .copy()
)

pegel_data = (
    pegel_data
    .sort_values("timestamp")
    .reset_index(drop=True)
    .copy()
)

feature_frames = [pegel_data]

# Hourly weather and water-level lags
for h in range(1, LAGS + 1):
    target_time = pegel_data["timestamp"] - pd.Timedelta(hours=h)

    matched_weather = (
        pd.DataFrame({"target_time": target_time})
        .merge(
            weather_data_hourly_pivoted,
            left_on="target_time",
            right_on="timestamp",
            how="left",
            validate="many_to_one",
        )
        .drop(columns=["target_time", "timestamp"])
    )

    matched_weather.columns = [
        f"{col}_lag_{h}h"
        for col in matched_weather.columns
    ]

    feature_frames.append(matched_weather)

    water_lag = pegel_data[["timestamp", "water_level"]].copy()
    water_lag["timestamp"] += pd.Timedelta(hours=h)

    water_lag = (
        pegel_data[["timestamp"]]
        .merge(
            water_lag,
            on="timestamp",
            how="left",
            validate="one_to_one",
        )
        .drop(columns="timestamp")
        .rename(columns={"water_level": f"water_level_lag_{h}h"})
    )

    feature_frames.append(water_lag)

    horb_lag = (
        pegel_data[["timestamp"]]
        .merge(
            horb_pegel_data[["timestamp", "horb_water_level"]].assign(
                timestamp=lambda frame: frame["timestamp"] + pd.Timedelta(hours=h)
            ),
            on="timestamp",
            how="left",
            validate="one_to_one",
        )
        .drop(columns="timestamp")
        .rename(columns={"horb_water_level": f"horb_water_level_lag_{h}h"})
    )

    feature_frames.append(horb_lag)

result = pd.concat(feature_frames, axis=1)

horb_indexed = (
    horb_pegel_data
    .set_index("timestamp")[["horb_water_level"]]
    .sort_index()
)
horb_rolling_24h = horb_indexed.rolling(window="24h", closed="left", min_periods=1).mean()

# Weather variables to aggregate.
# Exclude timestamp and any non-weather columns if present.
weather_columns = [
    col
    for col in weather_data_hourly_pivoted.columns
    if col != "timestamp" and col != "water_level"
]

weather_indexed = (
    weather_data_hourly_pivoted
    .set_index("timestamp")[weather_columns]
    .sort_index()
)

# A 24-hour trailing average, excluding the current timestamp.
weather_rolling_24h = (
    weather_indexed
    .rolling(
        window="24h",
        closed="left",
        min_periods=1,
    )
    .mean()
)

# Add offsets 0 through 12.
for offset in range(DAILY_OFFSETS + 1):
    lookup_time = result["timestamp"] - pd.Timedelta(days=offset)

    daily_lag = (
        pd.DataFrame({"lookup_time": lookup_time})
        .merge(
            weather_rolling_24h.reset_index(),
            left_on="lookup_time",
            right_on="timestamp",
            how="left",
            validate="many_to_one",
        )
        .drop(columns=["lookup_time", "timestamp"])
    )

    daily_lag.columns = [
        f"{col}_rolling_24h_offset_{offset}d"
        for col in daily_lag.columns
    ]

    feature_frames.append(daily_lag)

    horb_daily_lag = (
        pd.DataFrame({"lookup_time": lookup_time})
        .merge(
            horb_rolling_24h.reset_index(),
            left_on="lookup_time",
            right_on="timestamp",
            how="left",
            validate="many_to_one",
        )
        .drop(columns=["lookup_time", "timestamp"])
        .rename(columns={"horb_water_level": f"horb_water_level_rolling_24h_offset_{offset}d"})
    )

    feature_frames.append(horb_daily_lag)

# Rebuild the final result with the daily features included.
result = pd.concat(feature_frames, axis=1)

assert "_row_id" not in result.columns

assert all(
    f"horb_water_level_lag_{h}h" in result.columns
    for h in range(1, LAGS + 1)
)

assert all(
    f"horb_water_level_rolling_24h_offset_{offset}d" in result.columns
    for offset in range(DAILY_OFFSETS + 1)
)

assert all(
    f"water_level_lag_{h}h" in result.columns
    for h in range(1, LAGS + 1)
)

assert all(
    f"{col}_rolling_24h_offset_{offset}d" in result.columns
    for col in weather_columns
    for offset in range(DAILY_OFFSETS + 1)
)

assert len(result) == len(pegel_data)
assert result["timestamp"].equals(pegel_data["timestamp"])


In [23]:
# Check how many observations have missing values in the features
missing_values = result.isnull().sum()
print(missing_values)

timestamp                                           0
water_level                                         0
year                                                0
month                                               0
temperature_air_2m_02074_lag_1h                323281
                                                ...  
pressure_vapor_03278_rolling_24h_offset_11d    316957
pressure_vapor_03402_rolling_24h_offset_11d    308391
pressure_vapor_04931_rolling_24h_offset_11d     85086
pressure_vapor_06275_rolling_24h_offset_11d    323049
horb_water_level_rolling_24h_offset_11d           265
Length: 388, dtype: int64


In [24]:
min_time = result["timestamp"].min()
max_time = result["timestamp"].max()
diff = max_time - min_time
print(f"Time range of the dataset: {min_time} to {max_time}, which is a difference of {diff}.")
print(f"Total hours in the time range: {diff.days * 24 + 20}, which is {len(result)} rows in the dataset.")

Time range of the dataset: 1968-01-01 00:00:00 to 2026-11-24 11:00:00, which is a difference of 21512 days 11:00:00.
Total hours in the time range: 516308, which is 516296 rows in the dataset.


In [27]:
# To predict hourly over the next 2-3 days, you could use vectors as outputs, that contain the water_levels for the next 48-72h.
# Construct a dataframe that has a column water_levels_48/water_levels_72 that contains a 47/72 long pd array of water_levels.

import numpy as np

def add_forward_water_levels(features, horizon_hours):
    """Return rows with complete hourly water-level sequences through the horizon."""
    timestamps = pd.DatetimeIndex(features["timestamp"])
    water_by_time = features.drop_duplicates("timestamp").set_index("timestamp")["water_level"].sort_index()
    windows = np.column_stack([
        water_by_time.reindex(timestamps + pd.Timedelta(hours=offset)).to_numpy()
        for offset in range(horizon_hours + 1)
    ])
    complete = ~np.isnan(windows).any(axis=1)
    labeled = features.loc[complete, :].copy()
    labeled["water_levels"] = windows[complete].tolist()
    return labeled.reset_index(drop=True)

result_48h = add_forward_water_levels(result, 48)
result_72h = add_forward_water_levels(result, 72)
result_48h.to_parquet("../../data/final/pegel_weather_6_hourly_lags12_days_weather_lags_forecast_48h.parquet", index=False)
result_72h.to_parquet("../../data/final/pegel_weather_6_hourly_lags12_days_weather_lags_forecast_72h.parquet", index=False)
del result

assert "water_level" in result_48h.columns and "water_level" in result_72h.columns
assert np.allclose(result_48h["water_levels"].map(lambda levels: levels[0]), result_48h["water_level"])
assert np.allclose(result_72h["water_levels"].map(lambda levels: levels[0]), result_72h["water_level"])
assert len(result_48h) >= len(result_72h)
assert all(len(levels) == 49 for levels in result_48h["water_levels"])
assert all(len(levels) == 73 for levels in result_72h["water_levels"])
